入射角$\theta_s$から探査機角度$\alpha$を導く式
$$
\alpha = 2\theta_s - \sin^{-1} \{ \frac{R}{R+H}\times \sin{\theta_s}\}
$$
から、$\alpha=0.5,1.5,2.5 \dots 179.5$に対応する$\theta_s$をscipy.optimizeで導出

In [ ]:
import numpy as np
from scipy import optimize
import xarray as xr

def get_default_param(target):

    match target:
        case "moon":
            H_obs = 100e3       # 観測者の高度[m] (Kaguya)
            D_moon = 1*1e3      # 表層から地下構造までのレゴリスリス層(第一層)の厚さ[m]
            R_moon = 1737400.0  # 月の半径[m]
            e1 = 4.0            # レゴリス層(第一層)の比誘電率
            e2 = 8.0            # レゴリス層の下の地下構造(第二層)の比誘電率
            tandelta = 0.0125   # レゴリス層(第一層)の損失角

        case "ganymede":
            H_obs = 500e3       # 観測者の高度[m] (JUICE)
            D_moon = 1*1e3      # 表層から地下構造までのレゴリスリス層(第一層)の厚さ[m]
            R_moon = 5268000.0/2.0  # ガニメデの半径[m]
            e1 = 3.0            # 第一層の比誘電率
            e2 = 87.0           # 第二層の比誘電率
            tandelta = 0.0      # 第一層の損失角
    
    return e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta

def calc_alpha(ts, H, R):
    alpha = 2 * np.radians(ts) - np.arcsin(R/(R+H) * np.sin(np.radians(ts)))
    return np.degrees(alpha)

def calc_alpha_opt(ts, H, R, alpha_target):
    alpha = calc_alpha(ts, H, R)
    return alpha - alpha_target

In [ ]:
match_alpha = np.arange(0.5,180,1)

hs = [1200,2000,5000,10000]

target = "ganymede"
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = get_default_param(target)

In [ ]:
deriv_theta = np.zeros((len(match_alpha), len(hs)))

# hs (altitudes) x match_alpha (alpha targets)
for j, h in enumerate(hs):
    for i, ma in enumerate(match_alpha):
        d_th = optimize.fsolve(calc_alpha_opt, 1.0, args=(h, 1000, ma))
        deriv_theta[i, j] = float(d_th[0])

dth_da = xr.DataArray(deriv_theta, coords={"alpha": match_alpha, "H": hs}, dims=["alpha", "H"])
dth_da.name = "theta_s"
dth_da

In [ ]:
dth_da.to_netcdf("./nc_underground/alpha_to_theta_forhist.nc")